In [8]:
import os
import zipfile
import pandas as pd
import torch
import torchvision
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from torchvision.models import resnet101
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.models import resnet101
import torch.nn as nn
import torch.optim as optim


In [9]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"fominauliana","key":"4984975ff88bd6369e46087d5038daac"}'}

In [10]:
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
!pip install -q kaggle

!kaggle competitions download -c dog-breed-identification
with zipfile.ZipFile("dog-breed-identification.zip", 'r') as zip_ref:
    zip_ref.extractall("dog-breed-identification")

dog-breed-identification.zip: Skipping, found more recently modified local copy (use --force to force download)


In [11]:
labels = pd.read_csv('dog-breed-identification/labels.csv')
train_df, valid_df = train_test_split(
    labels,
    train_size=0.8,
    shuffle=True,
    stratify=labels['breed'],
    random_state=42
)

In [12]:
breed2idx = {breed: idx for idx, breed in enumerate(labels['breed'].value_counts().index)}
train_labels = train_df['breed'].map(breed2idx).values
valid_labels = valid_df['breed'].map(breed2idx).values
train_ids = train_df['id'].values
valid_ids = valid_df['id'].values

In [13]:
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.RandomRotation(10),
    torchvision.transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    torchvision.transforms.ToTensor()
])

val_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor()
])

In [14]:
class create_dataset(Dataset):
    def __init__(self, image_ids, labels, image_dir, transform=None):
        self.image_ids = image_ids
        self.labels = labels
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_ids[idx] + '.jpg')
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, torch.tensor(label).long()

In [15]:
train_dataset = create_dataset(train_ids, train_labels, 'dog-breed-identification/train', transform=train_transform)
val_dataset = create_dataset(valid_ids, valid_labels, 'dog-breed-identification/train', transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

resnet = resnet101(weights=None)
state_dict = torch.hub.load_state_dict_from_url(
    'https://download.pytorch.org/models/resnet101-63fe2227.pth'
)
resnet.load_state_dict(state_dict)

for param in resnet.parameters():
    param.requires_grad = False

resnet.fc = nn.Sequential(
    nn.Linear(resnet.fc.in_features, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.BatchNorm1d(256),
    nn.Linear(256, 120)
)

model = resnet.to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

In [17]:
n_epochs = 20
for epoch in range(1, n_epochs + 1):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = 0.0
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    acc = 100*correct/total
    print(f"Epoch {epoch}: Train Loss = {train_loss/len(train_loader):.4f}, "
          f"Val Loss = {val_loss/len(val_loader):.4f}, Accuracy = {acc:.2f}%")

torch.save(model.state_dict(), 'finetuned_resnet101.pth')

Epoch 1: Train Loss = 3.7007, Val Loss = 2.6358, Accuracy = 65.97%
Epoch 2: Train Loss = 2.4263, Val Loss = 1.8520, Accuracy = 74.62%
Epoch 3: Train Loss = 1.8339, Val Loss = 1.4426, Accuracy = 77.90%
Epoch 4: Train Loss = 1.4679, Val Loss = 1.1880, Accuracy = 79.71%
Epoch 5: Train Loss = 1.2402, Val Loss = 1.0345, Accuracy = 80.24%
Epoch 6: Train Loss = 1.0757, Val Loss = 0.9395, Accuracy = 80.88%
Epoch 7: Train Loss = 0.9637, Val Loss = 0.8768, Accuracy = 81.12%
Epoch 8: Train Loss = 0.8710, Val Loss = 0.8039, Accuracy = 81.86%
Epoch 9: Train Loss = 0.8132, Val Loss = 0.7672, Accuracy = 81.52%
Epoch 10: Train Loss = 0.7523, Val Loss = 0.7409, Accuracy = 81.91%
Epoch 11: Train Loss = 0.6937, Val Loss = 0.6938, Accuracy = 83.08%
Epoch 12: Train Loss = 0.6624, Val Loss = 0.6709, Accuracy = 82.69%
Epoch 13: Train Loss = 0.6262, Val Loss = 0.6859, Accuracy = 81.52%
Epoch 14: Train Loss = 0.6117, Val Loss = 0.6642, Accuracy = 81.52%
Epoch 15: Train Loss = 0.5791, Val Loss = 0.6537, Accurac

In [20]:
class TestImageDataset(Dataset):
    def __init__(self, image_ids, image_dir, transform=None):
        self.image_ids = image_ids
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.image_dir, img_id)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return img_id.replace('.jpg', ''), image


In [19]:
test_images = sorted(os.listdir('dog-breed-identification/test'))
test_dataset = TestImageDataset(test_images, 'dog-breed-identification/test', transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=2, pin_memory=True)

model.eval()
predictions = []

with torch.no_grad():
    for file_names, images in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        for fname, prob in zip(file_names, probs):
            predictions.append([fname] + list(prob))

columns = ['id'] + list(breed2idx.keys())
df_submission = pd.DataFrame(predictions, columns=columns)
df_submission['id'] = df_submission['id'].astype(str)
df_submission.to_csv('submission2.csv', index=False)

print("submission.csv загружен")

submission.csv загружен
